# Короткие артефакты ЭКГ: замена до фильтрации

По умолчанию `preprocess_ecg` обнаруживает редкие высокие выбросы на входе,
линейно заполняет разрешённые короткие участки и только затем включает фильтры.
На этой странице используем исходные отсчёты **до ресэмплинга**, чтобы выброс
не успел повлиять и на его антиалиас-фильтр. Источники `.e` и каталог читаются
без изменений. Записи и метки приступов остаются на исходной временной шкале.

Границы замен сохраняются. Внутри них настоящая ЭКГ неизвестна: интерполяция
убирает влияние большого выброса, но не восстанавливает потерянное измерение.
Принятое QC окно не является экспертно подтверждённой качественной ЭКГ.

**Ограничения по умолчанию:** кандидат >1000 мкВ и >8 типичных секундных пиков;
расширение до плеча выброса и по 20 мс с каждой стороны; заполнение не более
100 мс подряд и не более 1% окна по каждому каналу. Пики на границе,
насыщение АЦП и длинные дефекты остаются причинами отклонения.
Эти настройки не предназначены для удаления эпилептиформных спайков ЭЭГ.

Run All в ядре `.venv`. Это проверка реальных записей 85/210/31/44/202/15,
без генерации сигналов и обучения моделей. Запись 85 — тот же пик с картинки.


Для защиты повторяющихся комплексов дополнительно проверяются соседние секунды:
если вне кандидата есть пик хотя бы четверти его высоты, автоматической замены
не будет (`not_isolated`); дальнейшее решение принимает обычный QC.
Границы в отчёте относятся к исходной частоте и началу окна.

Почему крупный выброс обрабатывается до фильтрации: линейный фильтр может
распространить его влияние на соседние отсчёты. Такой эффект описан в
[документации EEGLAB](https://eeglab.org/tutorials/05_Preprocess/Filtering.html).


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "epi/__init__.py").exists())
sys.path.insert(0, str(ROOT))
from epi.preprocessing import preprocess_ecg, repair_ecg_spikes, ecg_quality_report
from scripts.validate_ecg_spike_repair import (
    OUT, CASES, reader_for, compare_case, case_row, plot_case, validate_examples,
)
OUT.mkdir(parents=True, exist_ok=True)


## Обычный вызов библиотеки уже включает обработку

`return_info=True` добавляет отчёт, но не включает отдельный режим: замена
коротких выбросов включена и при обычном `preprocess_ecg(raw, rate)`.
Для сравнения можно явно задать `repair_spikes=False`.


In [ ]:
reader = reader_for(85)
reader.select_segment(0)
raw = reader.read_channel("EKG", 405, 10)[None]
rate = reader.sampling_rate
filtered, info = preprocess_ecg(raw, rate,
    adc_limits=reader.channel_limits("EKG"), return_info=True)
qc = ecg_quality_report(raw, rate, filtered=filtered,
    adc_limits=reader.channel_limits("EKG"))
display(pd.DataFrame(info["intervals"]))
print("Заменено, мс:", 1000 * info["repaired_mask"].sum() / rate)
print("Доля окна, %:", 100 * info["repaired_fraction"])
print("QC после обработки:", qc["keep"], qc["reasons"])
assert info["repair_applied"] and not info["unrepaired_spike"]
assert filtered.shape == raw.shape and qc["keep"]


## Один и тот же реальный фрагмент: до/после

Оранжевым отмечены только заменённые исходные отсчёты. Нижняя кривая — разность
результатов двух способов обработки; это влияние изменения входа на фильтр,
а не ошибка относительно истинной ЭКГ (такого эталона здесь нет).


In [ ]:
case = compare_case(reader, 0, 405, 10)
fig = plot_case(case, "Запись 85, сегмент 0, 405–415 с", OUT / "spike_before_after.png")
plt.show()
display(pd.DataFrame([case_row(case, 85, 0)]))


## Контрольные записи и ограничения

Регулярные высокие зубцы не должны измениться. Насыщение АЦП должно по-прежнему
отклоняться по исходному сигналу. Дополнительно проверяем реальный пик на границе
окна, запрет превышения бюджета, сохранение отсчётов вне маски и обработку батчей.


In [ ]:
examples = validate_examples()
display(examples)


## Защита серии похожих пиков

В 300-секундном окне записи 29 короткий эпизод с несколькими высокими пиками
выделяется относительно типичной амплитуды всего окна. Одного глобального
порога недостаточно для автоматической замены. Сравнение с соседними пиками
оставляет эту серию без изменений; её происхождение здесь не классифицируется.


In [ ]:
cluster = compare_case(reader_for(29), 0, 2315, 300)
t = 2315 + np.arange(cluster["native"].shape[-1]) / cluster["rate"]
fig, ax = plt.subplots(figsize=(13, 3), layout="constrained")
ax.plot(t, cluster["native"][0], label="Исходный сигнал", lw=1)
ax.plot(t, cluster["repaired"][0], label="После проверки выбросов", lw=.8, linestyle="--")
ax.set(xlim=(2477.5, 2481), xlabel="Секунды от начала сегмента", ylabel="мкВ",
       title="Серия пиков сохраняется; решение об интерполяции учитывает соседей")
ax.legend()
ax.grid(alpha=.2)
fig.savefig(OUT / "cluster_preserved.png", dpi=130)
plt.show()
lo, hi = [int(round((s - 2315) * cluster["rate"])) for s in (2478, 2480)]
assert not cluster["info"]["repaired_mask"][:, lo:hi].any()
display(pd.DataFrame(cluster["info"]["intervals"]).head(10))


## Те же 400 окон из предыдущего аудита

Этот этап не запускает повторное чтение автоматически. Для пересчёта выполните
из корня `.venv/bin/python -m scripts.validate_ecg_spike_repair --audit-windows`.
Сравниваются прежние правила из `2e4e0ea` и текущая обработка; интервалы и метки
одинаковы. Сохраняется отбор по исходным дефектам, даже если выход выглядит гладким.
Статистика принятия не измеряет достоверность интерполяции или качество TCN.


In [ ]:
audit_path = OUT / "window_audit.csv"
if audit_path.exists():
    audit = pd.read_csv(audit_path)
    display(audit.groupby("label").agg(
        окон=("label", "size"), принято_до=("keep_before", "sum"),
        принято_после=("keep_after", "sum"),
        окон_с_заменами=("repaired_samples", lambda s: (s > 0).sum())))
    display(audit.loc[audit.keep_after != audit.keep_before].head(20))
else:
    print("Полный аудит ещё не выполнялся. Шесть реальных примеров проверены выше.")


## Использование в обучении

В модельном ноутбуке исходный сигнал сохраняется отдельно от подготовленного.
Маски представлены точными интервалами в `dev_spike_intervals.csv` и
`test_spike_intervals.csv`, с исходной частотой и привязкой к окну/сегменту.
Исходный QC объединяется с QC результата, затем индексы отбора одинаково
применяются к сигналам, меткам и группам пациентов. Нормализация идёт последней.

Текущая TCN не использует маску как отдельный вход: при допуске такого окна она
увидит интерполированный участок. Поэтому долю замен и сохранение положительных
окон надо анализировать отдельно; интерполяция не доказывает улучшение прогноза.
Для RR/HRV интервалы, пересекающие маску, нельзя считать восстановленными.
